# Global Earthquake Analysis
Final Python project: cleaning, manipulation, statistical analysis, and five visualizations.
Upload `global_earthquakes_seismic.csv` to Colab before running the notebook.

In [ ]:
from google.colab import files
uploaded = files.upload()
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import numpy as np
df = pd.read_csv(next(iter(uploaded)))
display(df.head())
print(df.shape)


## Cleaning
Remove duplicates and non-earthquakes, validate fields, convert dates, and save the cleaned CSV.

In [ ]:
import pandas as pd

# Load original earthquake data

print("Original shape:", df.shape)
print("Missing values:", df.isna().sum().sum())
print("Duplicate IDs:", df["id"].duplicated().sum())

# Clean blanks, duplicates, and non-earthquake events
df = df.replace(r"^\s*$", pd.NA, regex=True)
df = df.drop_duplicates().drop_duplicates(subset="id")
df = df.loc[df["type"].eq("earthquake")].copy()

# Correct data types and remove invalid critical records
df["time"] = pd.to_datetime(df["time"], unit="ms", utc=True, errors="coerce")
for col in ["latitude", "longitude", "depth", "mag"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna(subset=["id", "time", "latitude", "longitude", "depth", "mag"])
df = df.loc[df.latitude.between(-90, 90) & df.longitude.between(-180, 180) & (df.depth >= 0) & (df.mag >= 0)].copy()

# Add useful date fields, remove redundant narrative fields, sort and save
df["year"] = df.time.dt.year
df["month"] = df.time.dt.strftime("%Y-%m")
df["date"] = df.time.dt.strftime("%Y-%m-%d")
df = df.drop(columns=["evidence_span", "cot_rationale", "title"], errors="ignore")
df = df.sort_values("time").reset_index(drop=True)
df.to_csv("global_earthquakes_cleaned.csv", index=False)
print("Cleaned shape:", df.shape)



## Manipulation, statistics, and charts
Sort, filter, group, compute descriptive statistics and correlations, and save five charts.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("global_earthquakes_cleaned.csv")
df["time"] = pd.to_datetime(df["time"], utc=True, format="mixed")
out = Path("visualizations")
out.mkdir(exist_ok=True)
sns.set_theme(style="whitegrid")

# Sorting: find the largest events
print(df.sort_values("mag", ascending=False)[["place", "mag", "depth"]].head(10))
# Filtering: focus on stronger events
strong = df.loc[df["mag"] >= 6]
print("Magnitude 6+ events:", len(strong), "Percentage:", round(len(strong) / len(df) * 100, 2))
# Grouping: compare tectonic regions
region = "tectonic_boundary_zone"
print(df.groupby(region).agg(count=("id", "count"), mean_magnitude=("mag", "mean"), mean_depth=("depth", "mean")).round(2))
# Descriptive, correlation and distribution statistics
print(df[["mag", "depth"]].describe())
print("Magnitude mode:", df.mag.mode().iloc[0])
print("Magnitude skewness:", df.mag.skew())
print("Magnitude-depth correlation:", df.mag.corr(df.depth))

fig, ax = plt.subplots(figsize=(11, 6))
counts = df[region].value_counts()
sns.barplot(x=counts.values, y=counts.index, color="steelblue", ax=ax)
ax.set(title="Earthquakes by Tectonic Region", xlabel="Number of Earthquakes", ylabel="Tectonic Region")
fig.tight_layout(); fig.savefig(out / "earthquakes_by_region.png", dpi=160); plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 6))
sns.histplot(data=df, x="mag", bins=25, color="orange", edgecolor="black", ax=ax)
ax.set(title="Distribution of Earthquake Magnitudes", xlabel="Magnitude", ylabel="Number of Earthquakes")
fig.tight_layout(); fig.savefig(out / "magnitude_distribution.png", dpi=160); plt.close(fig)

fig, ax = plt.subplots(figsize=(10, 6))
sns.scatterplot(data=df, x="depth", y="mag", alpha=0.25, color="purple", label="Earthquakes", s=18, ax=ax)
ax.set(title="Earthquake Magnitude vs Depth", xlabel="Depth (km)", ylabel="Magnitude")
ax.legend(); fig.tight_layout(); fig.savefig(out / "magnitude_vs_depth.png", dpi=160); plt.close(fig)

monthly = df.set_index("time").resample("MS").size()
fig, ax = plt.subplots(figsize=(12, 6))
ax.plot(monthly.index, monthly.values, marker="o", color="green", label="Monthly earthquakes")
ax.set(title="Recorded Earthquakes by Month", xlabel="Month", ylabel="Number of Earthquakes")
ax.tick_params(axis="x", rotation=45); ax.legend()
fig.tight_layout(); fig.savefig(out / "earthquakes_over_time.png", dpi=160); plt.close(fig)

fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(df[["mag", "depth"]].corr(), annot=True, cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
ax.set_title("Magnitude and Depth Correlation")
fig.tight_layout(); fig.savefig(out / "correlation_heatmap.png", dpi=160); plt.close(fig)
print("Saved five charts to", out)



## Conclusion
The sample shows many events in the Pacific Ring of Fire and little linear relationship between magnitude and depth. Results should not be used to predict earthquakes.